# Training notebook for the SDG model

This notebook isolates the **training stage** of the synthetic data generator used in DIF-PI.

## What this notebook does

1. loads the **dunnhumby / Complete Journey** panel exported by `eda-complete-journey.ipynb`;
2. reuses the **DIF-PI SKU split logic** to obtain consistent `TRAIN_SKUs`, `TEST_SKUs`, and `CASE_SKU`;
3. builds tokenized seq2seq examples for the SDG model;
4. trains and saves the SDG checkpoint under `artifacts/models/`.

## Main output

- saved SDG checkpoint under `artifacts/models/sdg_chronos_t5_small_dunnhumby`

After the checkpoint is saved, use the validation notebook to load the model and run the validation experiments.


## 1) Environment and training configuration

This section sets the paths, the DIF-PI-compatible split parameters, and the SDG model training settings.

In [10]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
from pathlib import Path
import json
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.sdg import LLMSyntheticTimeSeriesGenerator, compute_difpi_sku_split

# DIF-PI-aligned paths
PANEL_PATH = REPO_ROOT / 'datasets' / 'processed' / 'difpi_pricing_demand_panel.csv'
OUT_DIR = REPO_ROOT / 'artifacts' / 'sdg_train'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Standardized columns
SKU_COL = 'StockCode'
TIME_COL = 'timestamp'
PRICE_COL = 'price'
DEMAND_COL = 'demand'

# DIF-PI protocol settings (matched to dif-pi.ipynb)
ELIGIBILITY_MODE = "adaptive"
MIN_HISTORY_DAYS_STRICT = 365
MIN_NONZERO_DAYS_STRICT = 60
MIN_HISTORY_DAYS_RELAXED = 120
MIN_NONZERO_DAYS_RELAXED = 30
SKU_HOLDOUT_ENABLED = True
TEST_SKU_FRAC = 0.20
SPLIT_SEED = 42
CASE_SKU_OVERRIDE = None
TARGET_ELIGIBLE_FRACTION = 0.8
HORIZON = 30
TRANSFORMER_SEQ_LEN = 30

# SDG settings
MODEL_NAME = "amazon/chronos-t5-small"
SDG_MODEL_DIR = REPO_ROOT / 'artifacts' / 'models' / 'sdg_chronos_t5_small_dunnhumby'
LOAD_EXISTING_MODEL = False
RUN_TRAINING = True

CONTEXT_LENGTH = 140
PREDICTION_LENGTH = HORIZON
STRIDE = 1
MAX_TRAIN_SKUS = None
INCLUDE_METADATA = False   # no raw SKU token
NUM_RETURN_SEQUENCES = 10
SEASONALITY_STRENGTH = 0.75

# Training settings
TRAIN_STEPS = 1500
BATCH_SIZE = 2
LEARNING_RATE = 2.5e-5
LORA_RANK = 32
LORA_ALPHA = 64

# Aggregate evaluation
MAX_EVAL_TEST_SKUS = None

# Optional HF upload
UPLOAD_TO_HF = False
HF_REPO_ID = "alexgrigoras/sdg_chronos_t5_small_dunnhumby"
HF_PRIVATE_REPO = False
HF_COMMIT_MESSAGE = "Upload SDG checkpoint"
HF_TOKEN = os.environ.get("HF_TOKEN")

print("PLATFORM:", platform.platform())
print("PANEL_PATH:", PANEL_PATH)
print("OUT_DIR:", OUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("SDG_MODEL_DIR:", SDG_MODEL_DIR)
print("HF_TOKEN_SET:", bool(HF_TOKEN))

PLATFORM: macOS-26.3.1-arm64-arm-64bit
PANEL_PATH: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/datasets/processed/difpi_pricing_demand_panel.csv
OUT_DIR: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/artifacts/sdg_train
MODEL_NAME: amazon/chronos-t5-small
SDG_MODEL_DIR: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/artifacts/models/sdg_chronos_t5_small_dunnhumby
HF_TOKEN_SET: False


## 2) Data loading

The processed panel exported by `eda-complete-journey.ipynb` is loaded here.

In [3]:
panel = pd.read_csv(PANEL_PATH)
panel[SKU_COL] = panel[SKU_COL].astype(str)
panel[TIME_COL] = pd.to_datetime(panel[TIME_COL], errors='raise')
panel = panel.sort_values([SKU_COL, TIME_COL]).reset_index(drop=True)

print(panel.shape)
display(panel.head())

(209633, 4)


,timestamp,StockCode,price,demand
0,2018-01-06,1000753,3.323333,3.0
1,2018-01-07,1000753,3.323333,0.0
2,2018-01-08,1000753,3.323333,0.0
3,2018-01-09,1000753,3.323333,0.0
4,2018-01-10,1000753,3.323333,0.0


## 3) DIF-PI-consistent SKU split

This section computes eligibility, train-test SKU partitioning, and the case SKU using the same DIF-PI split logic as the main framework.

In [4]:
split_info = compute_difpi_sku_split(
    panel_df=panel,
    sku_col=SKU_COL,
    time_col=TIME_COL,
    target_col=DEMAND_COL,
    eligibility_mode=ELIGIBILITY_MODE,
    min_history_days_strict=MIN_HISTORY_DAYS_STRICT,
    min_nonzero_days_strict=MIN_NONZERO_DAYS_STRICT,
    min_history_days_relaxed=MIN_HISTORY_DAYS_RELAXED,
    min_nonzero_days_relaxed=MIN_NONZERO_DAYS_RELAXED,
    transformer_seq_len=TRANSFORMER_SEQ_LEN,
    horizon=HORIZON,
    target_eligible_fraction=TARGET_ELIGIBLE_FRACTION,
    sku_holdout_enabled=SKU_HOLDOUT_ENABLED,
    test_sku_frac=TEST_SKU_FRAC,
    split_seed=SPLIT_SEED,
    case_sku_override=CASE_SKU_OVERRIDE,
)

sku_stats = split_info["sku_stats"]
eligible = split_info["eligible"]
TRAIN_SKUs = split_info["train_skus"]
TEST_SKUs = split_info["test_skus"]
CASE_SKU = split_info["case_sku"]

print(f"Eligibility mode: {ELIGIBILITY_MODE}")
print(f"Thresholds used: MIN_HISTORY_DAYS={split_info['min_history_days']}, MIN_NONZERO_DAYS={split_info['min_nonzero_days']}")
print(f"Eligible SKUs: {len(eligible)} | Train SKUs: {len(TRAIN_SKUs)} | Test SKUs: {len(TEST_SKUs)}")
print("CASE_SKU selected:", CASE_SKU)

pd.Series(TRAIN_SKUs, name="train_sku").to_csv(OUT_DIR / "train_skus.csv", index=False)
pd.Series(TEST_SKUs, name="test_sku").to_csv(OUT_DIR / "test_skus.csv", index=False)

display(sku_stats.describe(percentiles=[.1,.25,.5,.75,.9]))
display(eligible.head())

Eligibility mode: adaptive
Thresholds used: MIN_HISTORY_DAYS=365, MIN_NONZERO_DAYS=60
Eligible SKUs: 300 | Train SKUs: 240 | Test SKUs: 60
CASE_SKU selected: 9527487


,history_days,nonzero_days
count,300.000000,300.000000
mean,698.776667,506.603333
std,27.370127,127.896590
min,480.000000,112.000000
10%,690.000000,340.300000
25%,700.000000,421.250000
50%,706.000000,530.500000
75%,709.000000,599.250000
90%,710.000000,669.200000
max,711.000000,708.000000


,StockCode,history_days,nonzero_days
0,1082185,711,708
1,1029743,710,708
2,1106523,711,705
3,981760,711,705
4,995242,709,703


## 4) SDG model training

The SDG model is initialized and trained on the training SKU panel, then saved to the configured checkpoint directory.

In [5]:
train_skus_used = list(TRAIN_SKUs)
if MAX_TRAIN_SKUS is not None:
    train_skus_used = train_skus_used[:int(MAX_TRAIN_SKUS)]

sdg = LLMSyntheticTimeSeriesGenerator(
    model_name=MODEL_NAME,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    learning_rate=LEARNING_RATE,
    train_steps=TRAIN_STEPS,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    batch_size=BATCH_SIZE,
    seasonality_strength=SEASONALITY_STRENGTH,
)

MODEL_STATUS = "not_loaded"
MODEL_READY = False
MODEL_BACKEND = "unknown"
TRANSFORMERS_VERSION = None

try:
    import transformers
    TRANSFORMERS_VERSION = transformers.__version__
except Exception:
    TRANSFORMERS_VERSION = "not_available"

print("TRANSFORMERS_VERSION:", TRANSFORMERS_VERSION)

def _safe_load_sdg_checkpoint(model_dir):
    """Load either a full saved model or a PEFT/LoRA adapter checkpoint."""
    model_dir = Path(model_dir)
    cfg_path = model_dir / "sdg_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing SDG config: {cfg_path}")

    config = json.loads(cfg_path.read_text(encoding="utf-8"))
    obj = LLMSyntheticTimeSeriesGenerator(
        model_name=str(config.get("base_model_id") or config.get("model_name") or MODEL_NAME),
        context_length=int(config.get("context_length", CONTEXT_LENGTH)),
        prediction_length=int(config.get("prediction_length", PREDICTION_LENGTH)),
        num_bins=int(config.get("num_bins", 4094)),
        value_range=tuple(config.get("value_range", [-5.0, 5.0])),
        learning_rate=float(config.get("learning_rate", LEARNING_RATE)),
        train_steps=int(config.get("train_steps", TRAIN_STEPS)),
        lora_rank=int(config.get("lora_rank", LORA_RANK)),
        lora_alpha=int(config.get("lora_alpha", LORA_ALPHA)),
        batch_size=int(config.get("batch_size", BATCH_SIZE)),
        max_source_length=int(config.get("max_source_length", 768)),
        max_target_length=int(config.get("max_target_length", 256)),
        random_state=int(config.get("random_state", 42)),
        task_prefix=str(config.get("task_prefix", "generate synthetic retail demand future from historical context")),
        seasonality_strength=float(config.get("seasonality_strength", SEASONALITY_STRENGTH)),
    )

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    adapter_cfg_path = model_dir / "adapter_config.json"
    base_model_id = str(config.get("base_model_id") or config.get("model_name") or MODEL_NAME)

    if adapter_cfg_path.exists():
        from peft import PeftModel
        base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id)
        # The tokenizer may have been resized during fine-tuning.
        try:
            try:
                base_model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
            except TypeError:
                # Backward compatibility with older transformers versions.
                base_model.resize_token_embeddings(len(tokenizer))
        except Exception:
            pass
        model = PeftModel.from_pretrained(base_model, model_dir)
        backend_name = config.get("backend_name") or "lora"
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
        backend_name = config.get("backend_name") or "loaded"

    obj.model = model
    obj.tokenizer = tokenizer
    obj.backend_name = str(backend_name)
    obj.is_peft_model = bool(adapter_cfg_path.exists())
    obj.config["model_name"] = str(model_dir)
    loaded_base = config.get("base_model_id")
    if loaded_base:
        obj.config["base_model_id"] = str(loaded_base)

    ti = model_dir / "training_info.json"
    if ti.exists():
        try:
            obj.training_info = json.loads(ti.read_text(encoding="utf-8"))
            if obj.training_info.get("backend_name"):
                obj.backend_name = str(obj.training_info.get("backend_name"))
        except Exception:
            pass

    return obj

try:
    if LOAD_EXISTING_MODEL and SDG_MODEL_DIR.exists():
        sdg = _safe_load_sdg_checkpoint(SDG_MODEL_DIR)
        MODEL_STATUS = "loaded_existing_checkpoint"
        MODEL_READY = True
        MODEL_BACKEND = getattr(sdg, "backend_name", "loaded")
    elif RUN_TRAINING:
        train_df = sdg.build_training_dataframe(
            panel_df=panel,
            sku_col=SKU_COL,
            time_col=TIME_COL,
            target_col=DEMAND_COL,
            train_skus=train_skus_used,
            per_sku_train_frac=1.0,
            stride=STRIDE,
            include_metadata=INCLUDE_METADATA,
        )
        rng = np.random.default_rng(42)
        idx = np.arange(len(train_df))
        rng.shuffle(idx)
        n_eval = max(1, int(round(0.1 * len(idx))))
        eval_idx = idx[:n_eval]
        tr_idx = idx[n_eval:]
        eval_df = train_df.iloc[eval_idx].reset_index(drop=True)
        train_df_fit = train_df.iloc[tr_idx].reset_index(drop=True)

        training_info = sdg.fit(
            train_df=train_df_fit,
            eval_df=eval_df,
            output_dir=str(SDG_MODEL_DIR),
            logging_steps=25,
            save_steps=max(100, TRAIN_STEPS // 2),
        )
        sdg.save(
            output_dir=str(SDG_MODEL_DIR),
            push_to_hub=False,
        )
        MODEL_STATUS = "trained_and_saved"
        MODEL_READY = True
        MODEL_BACKEND = training_info.get("backend_name", getattr(sdg, "backend_name", "unknown"))
    else:
        MODEL_STATUS = "checkpoint_missing_local_dir" if not SDG_MODEL_DIR.exists() else "not_loaded"
        MODEL_READY = False
        MODEL_BACKEND = "unknown"
except Exception as exc:
    MODEL_STATUS = f"training_failed: {exc}"
    MODEL_READY = False
    MODEL_BACKEND = "unknown"
    raise

print("MODEL_STATUS:", MODEL_STATUS)
print("MODEL_READY:", MODEL_READY)
print("MODEL_BACKEND:", MODEL_BACKEND)
if not MODEL_READY:
    print("The notebook can still run the split / data cells, but generation requires a saved checkpoint or RUN_TRAINING=True.")

TRANSFORMERS_VERSION: 5.3.0


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/114782 [00:00<?, ? examples/s]

Map:   0%|          | 0/12754 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
25,37.886565,4.606209
50,37.611921,4.552639
75,37.096599,4.443545
100,36.298669,4.336967
125,35.875991,4.242682
150,34.716094,4.152118
175,34.438911,4.065175
200,33.897463,3.987807
225,34.360894,3.951249
250,33.365896,3.934506


/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


MODEL_STATUS: trained_and_saved
MODEL_READY: True
MODEL_BACKEND: lora


## 5) Optional Hugging Face upload

If `UPLOAD_TO_HF=True`, the saved checkpoint is pushed to your Hugging Face repo after training finishes.


In [11]:
if MODEL_READY and UPLOAD_TO_HF:
    if not HF_REPO_ID:
        raise ValueError("Set HF_REPO_ID before UPLOAD_TO_HF=True.")
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN in the environment before UPLOAD_TO_HF=True.")
    sdg.save(
        output_dir=str(SDG_MODEL_DIR),
        push_to_hub=True,
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
        private=HF_PRIVATE_REPO,
        commit_message=HF_COMMIT_MESSAGE,
    )
    print("Uploaded checkpoint to:", HF_REPO_ID)
else:
    print("UPLOAD_TO_HF:", UPLOAD_TO_HF)

/opt/homebrew/anaconda3/envs/scenario-generation-env/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded checkpoint to: alexgrigoras/sdg_chronos_t5_small_dunnhumby
